# SmolVLA fine-tuning checkpoints on the generated robot dataset

**Question.** SmolVLA (`lerobot/smolvla_base`, action expert only) is fine-tuned on the robot demos that were generated from real human clips (bowl on plate, real-size objects in LIBERO's scene). Does a longer-trained checkpoint do the task better, and does the policy's output get closer to the demonstrations?

| Test | What it measures | Caveat |
|---|---|---|
| **Closed loop** | The checkpoint drives the Panda in the `scaled` scene (real-size bowl and plate, other objects removed) from random layouts; success is LIBERO's own check. The same layout seeds are used for every checkpoint. | Few episodes per checkpoint: read the Wilson intervals, not the point estimates. |
| **Open loop** | Mean absolute error of the predicted 50-action chunk against the demonstrated one on sampled frames of the dataset (no simulator). | The frames are **training** frames (there is no held-out split), so this is a fit measure, not generalisation. |

The numbers come from `scripts/23_eval_checkpoints.py` (results in `results/smolvla_ckpts/`), the training run from `scripts/19_train_smolvla.py` (`outputs/e1_smolvla_ckpts/run`, log `outputs/e1_smolvla_ckpts/train.log`). This notebook shows whatever checkpoints exist when it is executed. Decision log: `docs/design_decisions.md` D-013.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from hurobo.utils.notebook import setup_notebook

ROOT = setup_notebook()

In [3]:
from hurobo.checkpoint_eval import (
    checkpoint_table,
    discover_checkpoints,
    display_table,
    parse_training_log,
    sample_size_note,
)
from hurobo.utils.viewer import checkpoint_progress_figure, training_loss_figure

RUN = ROOT / "outputs/e1_smolvla_ckpts/run"
RESULTS = ROOT / "results/smolvla_ckpts"
LOG_FREQ = 50

checkpoints = discover_checkpoints(RUN)
print("checkpoints on disk (steps):", [c.step for c in checkpoints])

checkpoints on disk (steps): [500, 1000, 1500, 2000, 2500, 3000]


## 1. Per-checkpoint numbers

In [4]:
table = checkpoint_table(RESULTS)
print(sample_size_note(table))
display_table(table)

Each closed-loop point is 10 episodes per checkpoint (same layout seeds for every checkpoint). Differences between checkpoints are within noise unless the Wilson intervals do not overlap.


,step,success (95 % Wilson),failed: bowl lifted,failed: never within 6 cm,mean steps (success),median closest eef-bowl (cm),open-loop frames,pos MAE,do-nothing pos MAE,gripper sign acc.
0,500,0/10 (0.00-0.28),1,9,NaN,14.6,100,0.113,0.137,0.92
1,1000,2/10 (0.06-0.51),1,7,410.0,12.1,100,0.103,0.137,0.93
2,1500,1/10 (0.02-0.40),1,8,215.0,7.9,100,0.096,0.137,0.95
3,2000,1/10 (0.02-0.40),2,7,335.0,11.4,100,0.085,0.137,0.95
4,2500,4/10 (0.17-0.69),0,6,321.0,8.3,100,0.088,0.137,0.95
5,3000,2/10 (0.06-0.51),3,8,292.0,8.6,100,0.084,0.137,0.96


`failed: bowl lifted` counts failed episodes in which the bowl rose more than 3 cm (grasped but not placed); `failed: never within 6 cm` counts failed episodes in which the end effector never came that close to the bowl centre (never reached it). Open-loop errors are in the environment's action units; the do-nothing column is the error of an all-zero chunk, so a policy only says something where it is below that.

## 2. Metrics against training step

In [5]:
if table.empty:
    print("no results yet")
else:
    checkpoint_progress_figure(table).show()

## 3. Training loss

In [6]:
losses = parse_training_log(RUN.parent / "train.log", LOG_FREQ)
if losses.empty:
    print("no training log found")
else:
    training_loss_figure(losses).show()
    print(f"{len(losses)} logged points, last loss {losses['loss'].iloc[-1]:.3f}")

60 logged points, last loss 0.244


## 4. Rollout videos

One closed-loop episode per checkpoint is saved as an mp4 in `results/smolvla_ckpts/videos/`.

In [7]:
sorted(p.name for p in (RESULTS / "videos").glob("*.mp4"))

['step000500__scaled__10__ep0.mp4',
 'step001000__scaled__10__ep0.mp4',
 'step001500__scaled__10__ep0.mp4',
 'step002000__scaled__10__ep0.mp4',
 'step002500__scaled__10__ep0.mp4',
 'step003000__scaled__10__ep0.mp4']